[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_13/3_stats.ipynb)

# Day 13: Statistics, A/B testing and product (medium)

**How to work:** Answer in two layers: the number (code) and one plain sentence a product manager understands. Name the method and its assumptions before computing.

**Today (27 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | The PM checks the dashboard every morning | medium | 7 |
| Q2 | Twenty metrics, one winner | medium | 7 |
| Q3 | A great first week | medium | 7 |
| Q4 | Not significant: so is it the same? (review) | medium | 6 |

**Handbook:** [Statistics, A/B testing and product](https://mahsa-agz.github.io/ds-handbook/#stats), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Data

- `ab` (Udacity e-commerce landing page test, real, 294,478 rows): `user_id`, `timestamp`, `group` (control / treatment), `landing_page` (old_page / new_page), `converted` (0/1). It has some rows where group and page do not match and some repeated users.

Q1 to Q3 use **simulated** experiments (made-up data with a known truth), because the point is to see how often a method fools you, which needs many repeated experiments. Q4 uses the real `ab` table.

## Setup

Run this cell once before the questions.

In [ ]:
# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
import statsmodels.formula.api as smf
import warnings
warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)
ab = pd.read_csv(data_path("ab_data.csv", "https://raw.githubusercontent.com/F-Zarian/Analyze_AB_test_results/main/ab_data.csv"))
print("loaded:", "ab", ab.shape)

---
### Q1. The PM checks the dashboard every morning

*medium, about 7 min*

Simulate 2,000 A/A tests (no true difference). Each test runs 20 days; each day 1,000 new users join each arm and convert with probability 0.10. Every day the PM runs a two-proportion z-test on all data so far.

(a) What share of A/A tests is 'significant' (|z| > 1.96) on day 20 only? What share is significant on **at least one** of the 20 days (stop at the first win)?

(b) What constant threshold on |z| would keep the false positive rate at 5% with 20 daily looks?

(c) Say it to a PM.

Follow-ups: name two valid ways to look early. Does peeking also bias the size of the effect you report?

In [ ]:
rng = np.random.default_rng(1)
sims, days, n_day, p = 2000, 20, 1000, 0.10
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: repeated significance tests on accumulating data. Pitfall: peeking (optional stopping) inflates the false positive rate. Simulate it as an A/A test.

</details>

<details><summary><b>Hint 2</b></summary>

1. `rng.binomial(n_day, p, size=(sims, days))` for each arm, then `cumsum(axis=1)`. 2. Cumulative n is `n_day * (day + 1)`. 3. Pooled z per day. 4. `(abs(z) > 1.96).any(axis=1).mean()` vs `abs(z[:, -1]) > 1.96`. 5. Threshold: 95th percentile of `abs(z).max(axis=1)`.

</details>

<details><summary><b>Solution</b></summary>

```python
rng = np.random.default_rng(1)
sims, days, n_day, p = 2000, 20, 1000, 0.10
xa = rng.binomial(n_day, p, size=(sims, days)).cumsum(axis=1)
xb = rng.binomial(n_day, p, size=(sims, days)).cumsum(axis=1)
n = n_day * np.arange(1, days + 1)                    # users per arm so far
pool = (xa + xb) / (2 * n)
z = (xb / n - xa / n) / np.sqrt(pool * (1 - pool) * 2 / n)
print("false positives, look once on day 20:", (abs(z[:, -1]) > 1.96).mean())          # 0.0445
print("false positives, stop at first p < 0.05:", (abs(z) > 1.96).any(axis=1).mean())  # 0.2325
print("threshold for 5% with 20 looks:", np.percentile(abs(z).max(axis=1), 95).round(2))  # 2.66
```

**Why:** Each look is another chance for noise to cross 1.96. With 20 looks about 23% of A/A tests are declared winners, almost 5 times the promised 5%. Keeping 5% overall needs a stricter bar at each look (about 2.66 here, close to the Pocock boundary for 20 looks), or a method built for peeking.

**Say it to a PM:** If we stop the test the first day the dashboard turns green, about 1 in 4 'wins' will be pure noise, so we either wait for the planned end date or use a sequential method that is designed for daily checks.

**Follow-ups:**

- *Valid early looks?* Group sequential designs with alpha spending (O'Brien-Fleming: very strict early, close to 1.96 at the end), always-valid p-values / mSPRT, or Bayesian monitoring with a pre-set rule. Early stopping for clear *harm* on guardrails is usually allowed and encouraged.
- *Effect size bias?* Yes. Tests stopped at an early peak stop exactly when noise pushed the estimate up, so reported effects of early stopped winners are too large (a winner's curse).

**Complexity:** O(sims * days) vectorised numpy, under a second.

**Common mistakes:** Simulating a single test (you need many to estimate a rate). Using 20 independent tests instead of cumulative data (the looks are correlated, which is why the rate is 23% and not `1 - 0.95^20` = 64%).

**Learn more:** [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls), [stats-bayesian-sequential](https://mahsa-agz.github.io/ds-handbook/#stats-bayesian-sequential)

</details>

---
### Q2. Twenty metrics, one winner

*medium, about 7 min*

(a) Simulate 2,000 A/A tests that each compare 20 independent metrics (500 users per arm, each metric standard normal). For each test, use Welch t-tests. What share of tests has at least one metric with p < 0.05? Compare with `1 - 0.95^20`.

(b) Apply Bonferroni (p < 0.05/20) and Benjamini-Hochberg (FDR 0.05, use `statsmodels.stats.multitest.multipletests`) to each test. What share still has a false winner?

(c) A real readout has these 8 p-values: `[0.001, 0.008, 0.012, 0.03, 0.04, 0.21, 0.45, 0.77]`. Which are significant with Bonferroni, Holm and BH at 0.05?

(d) Say it to a PM.

Follow-up: the PM slices the result by 'heavy vs light users, defined by sessions during the test'. What is wrong besides multiple testing?

In [ ]:
from statsmodels.stats.multitest import multipletests
rng = np.random.default_rng(2)
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: many metrics or segments tested at once. Pitfall: multiple comparisons; the family-wise error rate grows with the number of tests. Fixes: Bonferroni / Holm (FWER) or Benjamini-Hochberg (FDR).

</details>

<details><summary><b>Hint 2</b></summary>

1. Arrays shaped (sims, 20, 500) for each arm. 2. `stats.ttest_ind(a, b, axis=2, equal_var=False)`. 3. Any p < 0.05 per row. 4. `multipletests(p_row, method='bonferroni' | 'holm' | 'fdr_bh')[0]`.

</details>

<details><summary><b>Solution</b></summary>

```python
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(2)
sims, k, n = 2000, 20, 500
a = rng.standard_normal((sims, k, n))
b = rng.standard_normal((sims, k, n))
pv = stats.ttest_ind(a, b, axis=2, equal_var=False).pvalue        # shape (2000, 20)
print("any p < 0.05:", (pv < 0.05).any(axis=1).mean(), " theory:", round(1 - 0.95 ** 20, 3))
print("Bonferroni false winner:", (pv < 0.05 / k).any(axis=1).mean())
print("BH false winner:", np.mean([multipletests(r, 0.05, "fdr_bh")[0].any() for r in pv]))
# any p < 0.05: 0.65  theory: 0.642
# Bonferroni false winner: 0.0465
# BH false winner: 0.047

p8 = [0.001, 0.008, 0.012, 0.03, 0.04, 0.21, 0.45, 0.77]
for m in ["bonferroni", "holm", "fdr_bh"]:
    print(m, multipletests(p8, 0.05, m)[0].astype(int))
# bonferroni [1 0 0 0 0 0 0 0]
# holm [1 0 0 0 0 0 0 0]
# fdr_bh [1 1 1 0 0 0 0 0]
```

**Why:** With 20 null metrics, about 65% of tests show at least one 'significant' metric (theory 64%). Bonferroni and Holm control the chance of any false winner (FWER) at 5%. When *all* nulls are true, BH also gives about 5% (FDR equals FWER in that case); its advantage is power when some effects are real. In (c) Bonferroni uses 0.05/8 = 0.00625, so only 0.001 passes; Holm compares the 2nd smallest with 0.05/7 = 0.0071 and stops at 0.008. BH compares the i-th smallest with `0.05 * i / 8`: 0.012 <= 0.01875 passes, 0.03 > 0.025 and 0.04 > 0.03125 fail, so three discoveries.

**Say it to a PM:** We checked 8 metrics, so some 'wins' are expected by luck; after correcting for that, one metric is a solid win and two more are promising (expected to be mostly real), and the rest are noise.

**Follow-ups:**

- *Segments by in-test sessions?* Sessions during the test can be changed by the treatment, so 'heavy users' in treatment are not the same people as 'heavy users' in control. This is conditioning on a post-treatment variable and can create effects out of nothing. Segment only on attributes fixed before assignment (country, platform, pre-period activity).
- *Primary metric?* Pick one primary metric (and maybe a few guardrails) before the test; correct the rest or treat them as exploratory.

**Complexity:** O(sims * k * n) for the simulation (20 million numbers per arm, a few seconds).

**Common mistakes:** Using BH to claim 'no false positives' (it controls the expected *share* of false discoveries). Forgetting that slicing by segments is also multiple testing.

**Learn more:** [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls), [stats-hypothesis-tests](https://mahsa-agz.github.io/ds-handbook/#stats-hypothesis-tests)

</details>

---
### Q3. A great first week

*medium, about 7 min*

Simulate a 28-day test on a fixed panel of returning users: 20,000 per arm, each user visits every day, control clicks with probability 0.10 per day. The treatment lift on day `t` (0 to 27) is `0.01 + 0.03 * exp(-t / 3)` (a novelty bump that fades to a true long-run lift of 1 pp).

(a) Estimate the lift (pp) from week 1 only and from week 4 only.

(b) Regress the daily lift on day number. How would you detect novelty in real data without knowing the truth?

(c) Say it to a PM.

Follow-ups: what is the opposite pattern called and when does it happen? How do new users help?

In [ ]:
rng = np.random.default_rng(3)
n, days = 20000, 28
t = np.arange(days)
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: an effect that changes with time since exposure. Pitfall: novelty (or primacy) effect. Look at the lift by day or by days since first exposure.

</details>

<details><summary><b>Hint 2</b></summary>

1. Daily clicks per arm: `rng.binomial(n, p_day)`. 2. Daily lift = treatment rate minus control rate. 3. Average days 0 to 6 and days 21 to 27. 4. `np.polyfit(t, lift, 1)` for a slope.

</details>

<details><summary><b>Solution</b></summary>

```python
rng = np.random.default_rng(3)
n, days = 20000, 28
t = np.arange(days)
true_lift = 0.01 + 0.03 * np.exp(-t / 3)
ctl = rng.binomial(n, 0.10, days) / n
trt = rng.binomial(n, 0.10 + true_lift) / n
lift = trt - ctl
print(f"week 1 lift: {lift[:7].mean() * 100:.2f} pp")     # 2.29 pp
print(f"week 4 lift: {lift[21:].mean() * 100:.2f} pp")    # 0.87 pp
slope = np.polyfit(t, lift, 1)[0]
print(f"slope: {slope * 100:.3f} pp per day")             # -0.065 pp per day
```

**Why:** The first week mixes the real 1 pp lift with a curiosity bump, so it more than doubles the long-run effect (2.29 pp vs 0.87 pp, where the true long-run lift is 1 pp and the rest is noise). A clearly negative trend of the daily lift is the warning sign. In real data, plot the lift by calendar day and, better, by *days since first exposure* (which separates novelty from calendar events), and compare users who joined late with users who joined early.

**Say it to a PM:** Users clicked a lot more in the first week because the design was new; by week four the gain settled at about 1 point, and that is the number we should plan with.

**Follow-ups:**

- *Opposite pattern?* Primacy effect (change aversion): existing users first react badly to a change they must relearn, and the lift grows later. Common with navigation or layout changes.
- *New users?* New users have no old habit to compare against, so their effect has no novelty or primacy part. If new and existing users show the same lift late in the test, novelty is gone. A long term holdout answers it for good.

**Complexity:** O(days).

**Common mistakes:** Running a test for 3 days because the result is already significant. Using calendar day only (a holiday can look like novelty). Extending a test forever: users clear cookies and change devices, which also dilutes the effect.

**Learn more:** [stats-ab-pitfalls](https://mahsa-agz.github.io/ds-handbook/#stats-ab-pitfalls)

</details>

---
### Q4. Not significant: so is it the same? (review)

*medium, about 6 min*

Use the cleaned `ab` table (drop group/page mismatches, keep the first row per user; you did this on day 12). The difference was not significant.

(a) Compute the 95% CI of the difference in conversion (treatment minus control).

(b) The business says any change smaller than 0.5 pp in absolute value does not matter. Run an equivalence test (TOST): two one-sided z-tests against -0.5 pp and +0.5 pp. Report the larger of the two p-values.

(c) Say it to a PM.

Follow-up: why is the 90% CI (not 95%) the one that matches a TOST at alpha 0.05?

In [ ]:
# your code here

<details><summary><b>Hint 1</b></summary>

Signal: 'no significant difference' and the PM wants 'the same'. Method: confidence interval and an equivalence test (TOST) with a margin chosen before the test.

</details>

<details><summary><b>Hint 2</b></summary>

1. Clean as on day 12. 2. `se = sqrt(p_t(1-p_t)/n_t + p_c(1-p_c)/n_c)`. 3. Lower test: `z1 = (diff + m) / se`, `p1 = norm.sf(z1)`. Upper: `z2 = (diff - m) / se`, `p2 = norm.cdf(z2)`. 4. TOST p = max(p1, p2).

</details>

<details><summary><b>Solution</b></summary>

```python
ok = (ab.group == "treatment") == (ab.landing_page == "new_page")
clean = ab[ok].sort_values("timestamp").drop_duplicates("user_id")
g = clean.groupby("group").converted.agg(["mean", "count"])
(p_c, n_c), (p_t, n_t) = g.loc["control"], g.loc["treatment"]
diff = p_t - p_c
se = np.sqrt(p_t * (1 - p_t) / n_t + p_c * (1 - p_c) / n_c)
print(f"diff {diff * 100:.3f} pp, 95% CI [{(diff - 1.96 * se) * 100:.3f}, {(diff + 1.96 * se) * 100:.3f}] pp")
# diff -0.158 pp, 95% CI [-0.394, 0.078] pp
m = 0.005
p1 = stats.norm.sf((diff + m) / se)       # H0: diff <= -m
p2 = stats.norm.cdf((diff - m) / se)      # H0: diff >= +m
print(f"TOST p = {max(p1, p2):.2g}")      # TOST p = 0.0022
```

**Why:** The CI from -0.39 to +0.08 pp lies inside the plus or minus 0.5 pp margin, and the TOST p-value is 0.0022, so we can claim the pages are practically equivalent. That is a different, stronger statement than 'not significant', and it is only possible because the test was large (SE 0.12 pp).

**Say it to a PM:** The new page is not better, and we can say more: any difference is smaller than half a point either way, so if the new page is cheaper to maintain it is safe to switch.

**Follow-ups:**

- *Why 90%?* TOST runs two one-sided tests, each at 5%. Both pass exactly when the 90% two-sided CI (which leaves 5% in each tail) sits inside the margin.

**Complexity:** O(n log n) for the cleaning sort.

**Common mistakes:** Saying 'no effect' from p > 0.05. Choosing the equivalence margin after seeing the data. Using a margin so wide that anything is 'equivalent'.

**Learn more:** [stats-ab-analysis](https://mahsa-agz.github.io/ds-handbook/#stats-ab-analysis), [stats-confidence-intervals](https://mahsa-agz.github.io/ds-handbook/#stats-confidence-intervals)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [AI and ML](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_13/4_ai.ipynb)